In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
!pip install protobuf==3.20.3

In [1]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, f1_score, precision_score, recall_score
import warnings
warnings.filterwarnings("ignore")

# --- CRITICAL CONSTANTS ---
BATCH_SIZE = 8   
IMG_SIZE = 256
CROP_SIZE = 224 # Required for DenseNet121
# NOTE: Kaggle data path for the NIH Chest X-ray dataset (Tiny Subset)
DATA_PATH = '/kaggle/input/nih-chest-xrays/sample'
IMAGE_DIR = os.path.join(DATA_PATH, 'images')

all_labels = ['Atelectasis', 'Cardiomegaly', 'Effusion', 'Infiltration', 'Mass', 'Nodule', 'Pneumonia', 'Pneumothorax', 'Consolidation', 'Edema', 'Emphysema', 'Fibrosis', 'Pleural_Thickening', 'Hernia']

print(f"TensorFlow Version: {tf.__version__}")
print(f"Using BATCH_SIZE: {BATCH_SIZE}. Training will be slow on CPU, but stable.")

2025-11-23 17:32:47.154973: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1763919167.438561      48 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1763919167.508921      48 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

TensorFlow Version: 2.18.0
Using BATCH_SIZE: 8. Training will be slow on CPU, but stable.


In [2]:
# Cell 2: Load Data, Split, and Assign Manual Class Weights

# 1. Load Data and Multi-Label Encoding
# Ensure this path matches the one listed in your Kaggle Data panel.
# If your path is different (e.g., if you renamed the dataset), change 'nih-chest-xrays' below.
DATA_PATH = '/kaggle/input/sample/sample'
LABEL_FILE = os.path.join(DATA_PATH, 'sample_labels.csv')

# Load the label CSV
metadata_df = pd.read_csv(LABEL_FILE) 

# --- CRITICAL FILTER FIX: Filter data to only include images that were actually downloaded ---
IMAGE_DIR = os.path.join(DATA_PATH, 'images')
available_images = os.listdir(IMAGE_DIR)
metadata_df = metadata_df[metadata_df['Image Index'].isin(available_images)]

# Multi-Label Encoding
encoded_labels_df = metadata_df['Finding Labels'].str.get_dummies(sep='|')
if 'No Finding' in encoded_labels_df.columns:
    encoded_labels_df = encoded_labels_df.drop(columns=['No Finding'])
metadata_df = pd.concat([metadata_df, encoded_labels_df], axis=1)

# Ensure all 14 disease columns exist, filling with 0 if missing
for label in all_labels:
    if label not in metadata_df.columns:
        metadata_df[label] = 0

metadata_df['Image Path'] = metadata_df['Image Index'].apply(lambda x: os.path.join(IMAGE_DIR, x))

# 2. Data Splitting (Train/Val/Test) - Stratified for Tiny Subset stability
train_val_df, test_df = train_test_split(
    metadata_df, test_size=0.1, random_state=42, 
    stratify=metadata_df['Atelectasis'] 
)
train_df, val_df = train_test_split(
    train_val_df, test_size=(1/9), random_state=42,
    stratify=train_val_df['Atelectasis'] 
)

# 3. MANUAL CLASS WEIGHTS (Stabilized)
class_weights_for_fit = {
    i: weight for i, weight in enumerate([
        0.50, 1.00, 0.70, 0.60, 1.50, 1.40, 1.30, 1.60, 0.40, 2.00, 2.50, 3.50, 3.00, 10.00
    ])
}

print("\n--- Data Summary ---")
print(f"Training set: {len(train_df)} images")
print(f"Validation set: {len(val_df)} images")
print(f"Test set: {len(test_df)} images")
print("\nClass Weights Defined.")


--- Data Summary ---
Training set: 4484 images
Validation set: 561 images
Test set: 561 images

Class Weights Defined.


In [3]:
# Cell 4: Visualize Class Distribution (Histogram/Bar Chart)

import matplotlib.pyplot as plt
import seaborn as sns

# Assuming encoded_labels_df is available from Cell 2

# 1. Calculate the total number of positive occurrences for each label in the full dataset
class_counts = encoded_labels_df.sum(axis=0)

# 2. Filter out 'No Finding' as its count is usually much larger and skews the visualization
if 'No Finding' in class_counts.index:
    disease_counts = class_counts.drop('No Finding')
else:
    disease_counts = class_counts

# 3. Sort the counts in descending order for better visualization
disease_counts = disease_counts.sort_values(ascending=False)

# 4. Create the visualization
plt.figure(figsize=(12, 7))

sns.barplot(x=disease_counts.index, y=disease_counts.values, palette="viridis")

# Set the title and labels
plt.title('Distribution of Positive Findings (Excluding "No Finding")', fontsize=16)
plt.xlabel('Disease Finding Label', fontsize=14)
plt.ylabel('Number of Positive Instances (Images)', fontsize=14)

# Rotate x-axis labels for readability
plt.xticks(rotation=45, ha='right')
plt.tick_params(axis='x', labelsize=10)

# Add the count values on top of the bars
for i, count in enumerate(disease_counts.values):
    # Adjust position slightly above the bar
    plt.text(i, count + 1, str(count), ha='center', va='bottom', fontsize=10)

plt.tight_layout()

# Save the plot
plt.savefig('disease_distribution_histogram.png')
plt.close()

print("Disease count distribution histogram saved as 'disease_distribution_histogram.png'")

Disease count distribution histogram saved as 'disease_distribution_histogram.png'


In [ ]:
# Cell 3: Data Generator and Pipelines

class CheXpertDataGenerator(tf.keras.utils.Sequence):
    def __init__(self, df, labels, batch_size, img_size, shuffle=True):
        self.df = df
        self.labels = labels
        self.batch_size = batch_size
        self.img_size = img_size
        self.shuffle = shuffle
        self.indices = np.arange(len(self.df))
        if self.shuffle:
            np.random.shuffle(self.indices)

    def __len__(self):
        # We must return the floor to ensure batches are complete
        return int(np.floor(len(self.df) / self.batch_size))

    def __getitem__(self, index):
        indices = self.indices[index*self.batch_size:(index+1)*self.batch_size]
        batch_df = self.df.iloc[indices]
        
        X = np.empty((self.batch_size, CROP_SIZE, CROP_SIZE, 3))
        Y = batch_df[self.labels].values
        
        for i, path in enumerate(batch_df['Image Path']):
            # Load, resize, convert to array
            img = tf.keras.utils.load_img(path, target_size=(self.img_size, self.img_size), color_mode='grayscale')
            img = tf.keras.utils.img_to_array(img)
            
            # Convert grayscale to 3 channels for DenseNet and crop
            img = np.repeat(img, 3, axis=-1)
            img = tf.image.crop_to_bounding_box(img, 16, 16, CROP_SIZE, CROP_SIZE)
            img = img / 255.0 # Normalize
            
            X[i,] = img
            
        return X, Y

# Create Data Pipelines
train_ds = CheXpertDataGenerator(train_df, all_labels, BATCH_SIZE, IMG_SIZE, shuffle=True)
val_ds = CheXpertDataGenerator(val_df, all_labels, BATCH_SIZE, IMG_SIZE, shuffle=False)
test_ds = CheXpertDataGenerator(test_df, all_labels, BATCH_SIZE, IMG_SIZE, shuffle=False)

print("Data generators created.")

In [ ]:
# Cell 4: Model Definition

def create_model(num_classes=len(all_labels), input_shape=(CROP_SIZE, CROP_SIZE, 3)):
    # Load DenseNet121 base, freezing the weights
    base_model = tf.keras.applications.DenseNet121(
        include_top=False,
        weights='imagenet', # Use pre-trained weights
        input_shape=input_shape
    )
    base_model.trainable = False # Freeze base for Stage 1

    # Define the input layer
    inputs = tf.keras.Input(shape=input_shape)

    # Connect base model and apply global pooling
    x = base_model(inputs, training=False)
    x = tf.keras.layers.GlobalAveragePooling2D()(x)

    # Add custom classification head
    x = tf.keras.layers.Dense(512, activation='relu')(x)
    x = tf.keras.layers.Dropout(0.5)(x)
    
    # Final output layer (sigmoid for multi-label classification)
    outputs = tf.keras.layers.Dense(num_classes, activation='sigmoid')(x)

    # Create the final model
    model = tf.keras.Model(inputs, outputs)
    
    return model

print("Model creation function 'create_model' defined.")

In [ ]:
# Cell 5: Loss Function Definitions

# --- ZLPR CONSTANTS ---
LAMBDA_ZLPR = 0.05 

# Standard BCE loss
def bce_loss(y_true, y_pred):
    return tf.keras.losses.binary_crossentropy(y_true, y_pred, from_logits=False)

# --- ROBUST MULTI-LABEL FOCAL LOSS (alpha=0.75) ---
def focal_loss(y_true, y_pred, gamma=2.0, alpha=0.75):
    y_true = tf.cast(y_true, tf.float32)
    y_pred = tf.cast(y_pred, tf.float32)
    epsilon = tf.keras.backend.epsilon()
    y_pred = tf.clip_by_value(y_pred, epsilon, 1. - epsilon)
    
    p_t = tf.where(tf.equal(y_true, 1), y_pred, 1 - y_pred)
    modulating_factor = tf.pow(1. - p_t, gamma)
    alpha_factor = tf.where(tf.equal(y_true, 1), alpha, 1. - alpha)
    
    loss = -alpha_factor * modulating_factor * tf.math.log(p_t)
    
    return tf.reduce_mean(tf.reduce_sum(loss, axis=-1))

# --- ZLPR LOSS ---
def zlpr_loss(y_true, y_pred):
    y_true = tf.cast(y_true, tf.float32)
    y_pred = tf.cast(y_pred, tf.float32)
    bce = tf.keras.losses.binary_crossentropy(y_true, y_pred, from_logits=False)
    
    epsilon = tf.keras.backend.epsilon()
    y_pred_clipped = tf.clip_by_value(y_pred, epsilon, 1. - epsilon)
    
    # ZLPR Regularizer term
    zlpr_term = (1 - y_true) * tf.math.log(1 - y_pred_clipped) 
    zlpr_regularizer = -tf.reduce_mean(tf.reduce_sum(zlpr_term, axis=-1))
    
    return bce + LAMBDA_ZLPR * zlpr_regularizer

# --- FZLPR LOSS ---
def fzlpr_loss(y_true, y_pred):
    focal = focal_loss(y_true, y_pred, gamma=2.0, alpha=0.75) 
    
    y_true = tf.cast(y_true, tf.float32)
    y_pred = tf.cast(y_pred, tf.float32)
    epsilon = tf.keras.backend.epsilon()
    y_pred_clipped = tf.clip_by_value(y_pred, epsilon, 1. - epsilon)
    
    # ZLPR Regularizer term
    zlpr_term = (1 - y_true) * tf.math.log(1 - y_pred_clipped) 
    zlpr_regularizer = -tf.reduce_mean(tf.reduce_sum(zlpr_term, axis=-1))
    
    return focal + LAMBDA_ZLPR * zlpr_regularizer

In [ ]:
# Cell 6: Optimized Training Function

def train_and_evaluate(loss_function, run_name):
    print(f"\n--- Starting Experiment: {run_name} ---")
    
    # 1. Reset Model Weights (CRUCIAL STEP)
    tf.keras.backend.clear_session()
    model = create_model() 

    # 2. Stage 1: Train Head Layers Only (10 Epochs)
    print("Stage 1: Training Head Layers (10 Epochs)")
    
    auc_metric = tf.keras.metrics.AUC(multi_label=True, num_labels=len(all_labels), name='auc')
    prec_metric = tf.keras.metrics.Precision(name='precision')
    rec_metric = tf.keras.metrics.Recall(name='recall')

    # OPTIMIZED LR for Stability
    optimizer_stage1 = tf.keras.optimizers.AdamW(learning_rate=5e-4, weight_decay=4e-3)

    model.compile(
        optimizer=optimizer_stage1, 
        loss=loss_function, 
        metrics=[auc_metric, prec_metric, rec_metric]
    )

    history = model.fit(
        train_ds, validation_data=val_ds, epochs=10, verbose=1, 
        class_weight=class_weights_for_fit      
    )
    
    # --- Stage 2: Fine-tune Full Model (10 Epochs) ---
    print("\nStage 2: Fine-tuning Full Model (10 Epochs)")
    model.trainable = True # Unfreeze the entire model

    # OPTIMIZED LR for Stability
    optimizer_stage2 = tf.keras.optimizers.AdamW(learning_rate=1e-4, weight_decay=4e-3)
    model.compile(
        optimizer=optimizer_stage2, 
        loss=loss_function, 
        metrics=[auc_metric, prec_metric, rec_metric]
    )

    history_stage2 = model.fit(
        train_ds, validation_data=val_ds, epochs=10, verbose=1, 
        class_weight=class_weights_for_fit      
    )
    
    # --- Evaluation ---
    print("\nEvaluating Model on Validation Set (at 0.5 threshold)...")
    results = model.evaluate(val_ds, verbose=0)
    
    final_auc, final_prec, final_rec = results[1], results[2], results[3]
    final_f1 = (2 * final_prec * final_rec) / (final_prec + final_rec + tf.keras.backend.epsilon())
    
    print(f"Validation AUC: {final_auc:.4f}, Precision: {final_prec:.4f}, Recall: {final_rec:.4f}, F1-Score: {final_f1:.4f}")
    
    return {
        'loss_name': run_name,
        'val_auc': final_auc, 
        'val_precision': final_prec,
        'val_recall': final_rec,
        'val_f1': final_f1,
        'model': model
    }

In [ ]:
# Cell 7: Main Training Execution (Run This Once)

all_results = {}

# 1. Train Binary Crossentropy (BCE) - BASELINE
print("--- STARTING BCE BASELINE ---")
all_results['BCE'] = train_and_evaluate(bce_loss, run_name='BCE') 

# 2. Train Focal Loss 
print("\n--- STARTING FOCAL LOSS ---")
all_results['Focal Loss'] = train_and_evaluate(focal_loss, run_name='Focal Loss (γ=2.0, α=0.75)') 

# 3. Train ZLPR
print("\n--- STARTING ZLPR ---")
all_results['ZLPR'] = train_and_evaluate(zlpr_loss, run_name='ZLPR (λ=0.05)') 

# 4. Train FZLPR
print("\n--- STARTING FZLPR ---")
all_results['FZLPR'] = train_and_evaluate(fzlpr_loss, run_name='FZLPR (Focal + ZLPR)') 

print("\nAll four training runs are complete. Proceed to Cell 8 for final analysis.")

In [ ]:
import os
import tensorflow as tf
from tensorflow.keras.models import save_model
import pickle

# Check if all_results is defined
if 'all_results' not in locals():
    print("Error: The 'all_results' variable was not found. Cannot save models.")
else:
    os.makedirs('saved_models', exist_ok=True)
    save_data = {}

    for name, result in all_results.items():
        model = result['model']
        safe_name = name.replace(" ", "_").replace("(", "").replace(")", "").replace("=", "_")
        file_path = os.path.join('saved_models', f'{safe_name}.h5')
        
        try:
            # Save the Keras model (architecture and weights)
            model.save(file_path)
            
            # Store necessary metrics/history for later reconstruction
            save_data[name] = {
                'model_path': file_path,
                'loss_name': result['loss_name'],
                'val_auc': result['val_auc'],
                'history': result['model'].history.history
            }
            print(f"Model saved: {name}")
            
        except Exception as e:
            print(f"Error saving model {name}: {e}. Skipping.")

    # Save the dictionary containing paths and metrics
    with open('saved_models/analysis_data.pkl', 'wb') as f:
        pickle.dump(save_data, f)
        
    print("\nSUCCESS: All models and analysis data saved to the 'saved_models' directory.")

In [ ]:
# --- FINAL SUCCESSFUL CODE (Fixing the Pandas Printing Error) ---

import os
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import load_model
from sklearn.metrics import f1_score, precision_score, recall_score
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
import pickle
import gc 
from functools import partial

# --- CRITICAL FIX: Custom Loss and Metric Definitions (Required for load_model and recompile) ---

# 1. FocalLoss 
class FocalLoss(tf.keras.losses.Loss):
    def __init__(self, gamma=2.0, alpha=0.75, name='focal_loss', **kwargs):
        super().__init__(name=name, **kwargs)
        self.gamma = tf.constant(gamma, dtype=tf.float32)
        self.alpha = tf.constant(alpha, dtype=tf.float32)
        self.bce = tf.keras.losses.BinaryCrossentropy(from_logits=False, reduction=tf.keras.losses.Reduction.NONE)
    def call(self, y_true, y_pred):
        bce = self.bce(y_true, y_pred)
        p_t = (y_true * y_pred) + ((1 - y_true) * (1 - y_pred))
        alpha_factor = (y_true * self.alpha) + ((1 - y_true) * (1 - self.alpha))
        modulating_factor = tf.pow(1.0 - p_t, self.gamma)
        return tf.reduce_mean(alpha_factor * modulating_factor * bce)
    def get_config(self):
        config = super().get_config()
        config.update({'gamma': self.gamma.numpy(), 'alpha': self.alpha.numpy()})
        return config

# 2. ZLPRLoss 
class ZLPRLoss(tf.keras.losses.Loss):
    def __init__(self, lambda_val=0.05, name='zlpr_loss', **kwargs):
        super().__init__(name=name, **kwargs)
        self.lambda_val = lambda_val
    def call(self, y_true, y_pred):
        y_true = tf.cast(y_true, tf.float32)
        y_pred = tf.clip_by_value(y_pred, 1e-7, 1.0 - 1e-7)
        bce = -y_true * tf.math.log(y_pred) - (1 - y_true) * tf.math.log(1 - y_pred)
        zplr = tf.abs(tf.math.log(y_pred) - tf.math.log(1 - y_pred)) * (1 - 2 * y_true)
        return tf.reduce_mean(bce + self.lambda_val * zplr)
    def get_config(self):
        config = super().get_config()
        config.update({'lambda_val': self.lambda_val})
        return config
        
# 3. FZLPRLoss
class FZLPRLoss(tf.keras.losses.Loss):
    def __init__(self, gamma=2.0, alpha=0.75, lambda_val=0.05, name='fzlpr_loss', **kwargs):
        super().__init__(name=name, **kwargs)
        self.focal_loss = FocalLoss(gamma=gamma, alpha=alpha)
        self.zlpr_loss = ZLPRLoss(lambda_val=lambda_val)
        self.gamma = gamma
        self.alpha = alpha
        self.lambda_val = lambda_val
    def call(self, y_true, y_pred):
        return self.focal_loss(y_true, y_pred) + self.zlpr_loss(y_true, y_pred)
    def get_config(self):
        config = super().get_config()
        config.update({'gamma': self.gamma, 'alpha': self.alpha, 'lambda_val': self.lambda_val})
        return config

# Dictionary of all custom objects for loading
CUSTOM_OBJECTS = {
    'FocalLoss': FocalLoss,
    'ZLPRLoss': ZLPRLoss,
    'FZLPRLoss': FZLPRLoss,
    'AUC': tf.keras.metrics.AUC 
}
# ------------------------------------------------------------------

# --- CRITICAL FIX: REPLACE THIS PLACEHOLDER WITH YOUR ACTUAL PICKLE PATH ---
CORRECT_PICKLE_PATH = '/kaggle/working/saved_models/analysis_data.pkl' 
# ------------------------------------------------------------------


# 1. Re-define essential constants and data paths
all_labels = ['Atelectasis', 'Cardiomegaly', 'Effusion', 'Infiltration', 'Mass', 'Nodule', 'Pneumonia', 'Pneumothorax', 'Consolidation', 'Edema', 'Emphysema', 'Fibrosis', 'Pleural_Thickening', 'Hernia']
BATCH_SIZE = 8
IMG_SIZE = 256
CROP_SIZE = 224 
DATA_PATH = '/kaggle/input/sample/sample'
IMAGE_DIR = os.path.join(DATA_PATH, 'images')

# 2. Define the Preprocessing Function
def load_and_preprocess_image(image_path, label):
    img_path = tf.strings.join([IMAGE_DIR, image_path], separator=os.sep)
    img = tf.io.read_file(img_path)
    img = tf.image.decode_jpeg(img, channels=1) 
    img = tf.image.resize(img, [IMG_SIZE, IMG_SIZE])
    img = tf.image.grayscale_to_rgb(img) 
    img = tf.image.crop_to_bounding_box(img, 16, 16, CROP_SIZE, CROP_SIZE)
    img = img / 255.0
    return img, label

# 3. Load Data, Encode Columns, and Split
print("--- Reloading and Encoding Data ---")
LABEL_FILE = os.path.join(DATA_PATH, 'sample_labels.csv')
metadata_df = pd.read_csv(LABEL_FILE) 

encoded_labels_df = metadata_df['Finding Labels'].str.get_dummies(sep='|')
if 'No Finding' in encoded_labels_df.columns:
    encoded_labels_df = encoded_labels_df.drop(columns=['No Finding'])
metadata_df = pd.concat([metadata_df, encoded_labels_df], axis=1)
for label in all_labels:
    if label not in metadata_df.columns:
        metadata_df[label] = 0

train_val_df, test_df = train_test_split(
    metadata_df, test_size=0.1, random_state=42, 
    stratify=metadata_df['Atelectasis'] 
)
train_df, val_df = train_test_split(
    train_val_df, test_size=(1/9), random_state=42,
    stratify=train_val_df['Atelectasis'] 
)

# 4. Filter missing files and Create the HIGH-SPEED tf.data Validation Dataset
val_df_reduced = val_df.iloc[:len(val_df) // 2].copy()

# FILE FILTERING LOGIC
val_df_reduced['full_path'] = val_df_reduced['Image Index'].apply(lambda x: os.path.join(IMAGE_DIR, x))
val_df_reduced['exists'] = val_df_reduced['full_path'].apply(os.path.exists)

original_count = len(val_df_reduced)
val_df_reduced = val_df_reduced[val_df_reduced['exists']].drop(columns=['full_path', 'exists'])

val_paths = tf.constant(val_df_reduced['Image Index'].values)
val_labels = tf.constant(val_df_reduced[all_labels].values, dtype=tf.float32)
val_ds_fast = tf.data.Dataset.from_tensor_slices((val_paths, val_labels))
val_ds_fast = val_ds_fast.map(load_and_preprocess_image, num_parallel_calls=tf.data.AUTOTUNE)
val_ds_fast = val_ds_fast.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

y_true_val_list = []
for _, Y in val_ds_fast: 
    y_true_val_list.append(Y)
y_true_val = np.concatenate(y_true_val_list, axis=0) 

print(f"Data set size for prediction: {len(y_true_val)} samples.")

# 5. Load Saved Models and Run Prediction on GPU
if not os.path.exists(CORRECT_PICKLE_PATH):
    print(f"\nFATAL ERROR: The pickle file was not found at {CORRECT_PICKLE_PATH}.")
else:
    with open(CORRECT_PICKLE_PATH, 'rb') as f:
        save_data = pickle.load(f)
        
    model_predictions = {}
    all_results = {} 

    print("\n--- Loading Models onto GPU and Predicting (Should be sub-minute) ---")
    device_name = "/GPU:0" if tf.config.list_physical_devices('GPU') else "/CPU:0"
    optimizer = tf.keras.optimizers.AdamW(learning_rate=1e-3, weight_decay=4e-3)
    # The AUC metric must match the one used during training
    auc_metric = tf.keras.metrics.AUC(multi_label=True, num_labels=len(all_labels), name='auc')

    for name, data in save_data.items():
        print(f"Starting prediction for {name}...")
        
        # --- FIX 1: LOAD WITHOUT COMPILATION ---
        model = load_model(data['model_path'], custom_objects=CUSTOM_OBJECTS, compile=False)
        
        # --- FIX 2: MANUALLY RECOMPILE WITH CORRECT LOSS ---
        loss_name = data['loss_name']
        if loss_name == 'BCE':
            loss_fn = tf.keras.losses.BinaryCrossentropy()
        elif loss_name == 'FocalLoss':
            # Use the saved parameters if available, otherwise use default
            params = data.get('loss_params', {}) 
            loss_fn = FocalLoss(**params)
        elif loss_name == 'ZLPRLoss':
            params = data.get('loss_params', {})
            loss_fn = ZLPRLoss(**params)
        elif loss_name == 'FZLPRLoss':
            params = data.get('loss_params', {})
            loss_fn = FZLPRLoss(**params)
        else:
            loss_fn = tf.keras.losses.BinaryCrossentropy() # Fallback

        model.compile(optimizer=optimizer, loss=loss_fn, metrics=[auc_metric])

        # Run prediction on the GPU
        with tf.device(device_name):
            y_pred_val = model.predict(val_ds_fast, verbose=0)
            
        model_predictions[name] = y_pred_val
        
        all_results[name] = {
            'model': model, 
            'loss_name': data['loss_name'],
            'val_auc': data['val_auc'],
        }
        del model
        gc.collect()
        tf.keras.backend.clear_session()

    print("Predictions complete. Proceeding to Threshold Optimization (FAST).")


    # --- 6. Iterating and Optimizing Threshold (Final Metrics) ---
    final_optimized_metrics = {}
    thresholds = np.linspace(0.01, 0.50, 50) 

    for name in model_predictions.keys():
        best_f1_val = 0
        optimal_threshold = 0.5
        
        for t in thresholds:
            y_pred_binary = (model_predictions[name] > t).astype(int)
            f1 = f1_score(y_true_val, y_pred_binary, average='micro', zero_division=0)
            
            if f1 > best_f1_val:
                best_f1_val = f1
                optimal_threshold = t
                
        y_pred_optimal = (model_predictions[name] > optimal_threshold).astype(int)
        
        final_optimized_metrics[name] = {
            'Loss': all_results[name]['loss_name'],
            'Opt_Threshold': optimal_threshold,
            'Val_AUC': float(all_results[name]['val_auc']),
            'Opt_F1': f1_score(y_true_val, y_pred_optimal, average='micro', zero_division=0),
            'Opt_Precision': precision_score(y_true_val, y_pred_optimal, average='micro', zero_division=0),
            'Opt_Recall': recall_score(y_true_val, y_pred_optimal, average='micro', zero_division=0),
        }

    # --- 7. Comparison Table (Optimized Metrics) ---
    comparison_df = pd.DataFrame(final_optimized_metrics).T.sort_values(by='Opt_F1', ascending=False)
    # --- FINAL FIX: ROUND AND REMOVE FLOAT_FORMAT FOR PRINTING ---
    comparison_df = comparison_df.round(4)
    best_loss_name = comparison_df.index[0]

    print("\n\n=============================================================================")
    print("  FINAL COMPARISON (Metrics at OPTIMAL THRESHOLD - F1 Maximized) ")
    print("=============================================================================")
    display_cols = ['Loss', 'Opt_F1', 'Opt_Precision', 'Opt_Recall', 'Val_AUC', 'Opt_Threshold']
    # Use to_string() without explicit float_format
    print(comparison_df[display_cols].to_string())
    print("=============================================================================")
    print(f"The best performing model (by F1-Score) is: {best_loss_name}")

    # --- 8. History Plot ---
    history = save_data[best_loss_name]['history']

    plt.figure(figsize=(14, 6))

    # Plot Validation AUC
    plt.subplot(1, 2, 1)
    plt.plot(history['auc'], label='Train AUC', color='blue')
    plt.plot(history['val_auc'], label='Validation AUC', color='red')
    plt.title(f'Training & Validation AUC for Best Model: {best_loss_name}')
    plt.xlabel('Epoch')
    plt.ylabel('AUC')
    plt.legend()
    

    # Plot Validation Loss
    plt.subplot(1, 2, 2)
    plt.plot(history['loss'], label='Train Loss', color='blue')
    plt.plot(history['val_loss'], label='Validation Loss', color='red')
    plt.title(f'Training & Validation Loss for Best Model: {best_loss_name}')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.legend()
    

    plt.tight_layout()
    plt.show()

In [ ]:
import os
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import load_model
from sklearn.metrics import f1_score, precision_score, recall_score
import matplotlib.pyplot as plt
import pickle

# --- CRITICAL FIX: REPLACE THIS PLACEHOLDER WITH YOUR ACTUAL PICKLE PATH ---
CORRECT_PICKLE_PATH = '/kaggle/working/saved_models/analysis_data.pkl' 

# Load data to inspect keys
if os.path.exists(CORRECT_PICKLE_PATH):
    with open(CORRECT_PICKLE_PATH, 'rb') as f:
        save_data = pickle.load(f)

    # The best performing model from your final comparison
    best_loss_name = 'Focal Loss'
    history = save_data[best_loss_name]['history']

    # New code to find the correct AUC key
    auc_key = 'auc'
    val_auc_key = 'val_auc'
    
    # Check if the simple keys exist
    if auc_key not in history:
        # Search for a key ending in 'auc'
        for key in history.keys():
            if key.endswith('auc') and not key.startswith('val_'):
                auc_key = key
            if key.endswith('auc') and key.startswith('val_'):
                val_auc_key = key
                
    print(f"Detected AUC key for plotting: {auc_key}")
    
    # --- 8. History Plot ---
    plt.figure(figsize=(14, 6))

    # Plot Validation AUC
    plt.subplot(1, 2, 1)
    plt.plot(history[auc_key], label='Train AUC', color='blue')
    plt.plot(history[val_auc_key], label='Validation AUC', color='red')
    plt.title(f'Training & Validation AUC for Best Model: {best_loss_name}')
    plt.xlabel('Epoch')
    plt.ylabel('AUC')
    plt.legend()
    plt.savefig('auc_plot.png') 

    # Plot Validation Loss
    plt.subplot(1, 2, 2)
    plt.plot(history['loss'], label='Train Loss', color='blue')
    plt.plot(history['val_loss'], label='Validation Loss', color='red')
    plt.title(f'Training & Validation Loss for Best Model: {best_loss_name}')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.legend()
    plt.savefig('loss_plot.png') 
    
    plt.close()